In [3]:
import pandas as pd
from pyvis.network import Network
import IPython
from pathlib import Path

def converter_triplas_para_nos_arestas(df_triplas, col_e1='e1_text', col_t1='e1_type', 
                                       col_e2='e2_text', col_t2='e2_type', col_rel='relation_label'):
    # Filtra relações e entidades inválidas preditas pelos modelos
    df_valido = df_triplas[
        (df_triplas[col_rel] != 'NO_RELATION') & 
        (df_triplas[col_t1] != 'NOT_ENTITY') & 
        (df_triplas[col_t2] != 'NOT_ENTITY')
    ].copy()

    # Extrai nós únicos
    nos_origem = df_valido[[col_e1, col_t1]].rename(columns={col_e1: 'label_normalizado', col_t1: 'node_type'})
    nos_destino = df_valido[[col_e2, col_t2]].rename(columns={col_e2: 'label_normalizado', col_t2: 'node_type'})
    
    nodes_df = pd.concat([nos_origem, nos_destino], ignore_index=True)
    nodes_df['label_normalizado'] = nodes_df['label_normalizado'].astype(str).str.lower().str.strip()
    nodes_df = nodes_df.drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)
    nodes_df.insert(0, 'node_id', [f"N_{i+1:03d}" for i in range(len(nodes_df))])

    # Extrai arestas únicas
    edges_df = df_valido[[col_e1, col_e2, col_rel]].rename(columns={
        col_e1: 'source_label', 
        col_e2: 'target_label', 
        col_rel: 'relation'
    }).copy()
    edges_df['source_label'] = edges_df['source_label'].astype(str).str.lower().str.strip()
    edges_df['target_label'] = edges_df['target_label'].astype(str).str.lower().str.strip()
    edges_df = edges_df.drop_duplicates(subset=['source_label', 'target_label', 'relation']).reset_index(drop=True)
    edges_df.insert(0, 'edge_id', [f"E_{i+1:03d}" for i in range(len(edges_df))])

    return nodes_df, edges_df

def plotar_grafo_projeto2(nodes_df, edges_df, output_html="knowledge_graph.html"):
    net = Network(notebook=True, directed=True, cdn_resources='remote', height="800px", width="100%")
    
    color_map = {
        'PATIENT': '#4CAF50', 'Patient': '#4CAF50',
        'SYMPTOM': '#FF7043', 'DISEASE': '#E53935', 'DIAGNOSIS': '#E53935',
        'FINDING': '#42A5F5', 'MedicalConcept': '#42A5F5',
        'PROCEDURE': '#FFEE58', 'Procedure/Exam': '#FFEE58',
        'VALUE': '#BDBDBD', 'ExamResult': '#BDBDBD'
    }

    for _, row in nodes_df.iterrows():
        node_id = row['label_normalizado']
        node_type = row['node_type']
        net.add_node(
            node_id, label=node_id, 
            title=f"ID: {row.get('node_id', '')} | Tipo: {node_type}", 
            color=color_map.get(node_type, '#d9d9d9')
        )

    existing_nodes = set(net.get_nodes())
    for _, row in edges_df.iterrows():
        if row['source_label'] in existing_nodes and row['target_label'] in existing_nodes:
            net.add_edge(
                row['source_label'], row['target_label'], 
                title=row['relation'], label=row['relation'], color="#808080"
            )

    net.set_options("""
    var options = {
      "physics": {
        "forceAtlas2Based": {"gravitationalConstant": -60, "centralGravity": 0.01, "springLength": 110, "springConstant": 0.08},
        "minVelocity": 0.75, "solver": "forceAtlas2Based"
      },
      "edges": {"font": {"size": 10, "align": "middle"}, "arrows": {"to": {"enabled": true, "scaleFactor": 0.5}}}
    }
    """)
    net.show(output_html)
    return IPython.display.HTML(filename=output_html)

In [4]:
# A Pipeline 1 já salva os arquivos nodes.csv e edges.csv diretamente na pasta processed
PROCESSED_DIR = Path('../../data/processed')

nodes_p1 = pd.read_csv(PROCESSED_DIR / 'nodes.csv')
edges_p1 = pd.read_csv(PROCESSED_DIR / 'edges.csv')

# Garante que as colunas do CSV tenham os nomes esperados pela função de plotagem
if 'e1_text' in edges_p1.columns:
    edges_p1 = edges_p1.rename(columns={'e1_text': 'source_label', 'e2_text': 'target_label', 'relation_label': 'relation'})
elif 'source' in edges_p1.columns:
    edges_p1 = edges_p1.rename(columns={'source': 'source_label', 'target': 'target_label'})

if 'label_normalizado' not in nodes_p1.columns and 'label' in nodes_p1.columns:
    nodes_p1 = nodes_p1.rename(columns={'label': 'label_normalizado'})
elif 'label_normalizado' not in nodes_p1.columns and 'e1_text' in nodes_p1.columns:
    nodes_p1 = nodes_p1.rename(columns={'e1_text': 'label_normalizado'})

# Cria a pasta de destino automaticamente se ela não existir
output_dir = Path('../../assets/html')
output_dir.mkdir(parents=True, exist_ok=True)
output_file_p1 = str(output_dir / "knowledge_graph_pipeline1.html")

print("Plotando Grafo da Pipeline 1 (Rede Neural MLP)...")
plotar_grafo_projeto2(nodes_p1, edges_p1, output_html=output_file_p1)

Plotando Grafo da Pipeline 1 (Rede Neural MLP)...
../../assets/html/knowledge_graph_pipeline1.html


In [5]:
# A Pipeline 2 salva no formato de triplas no CSV graph.csv
PIPELINE2_DIR = Path('../pipeline2/outputs')

df_p2 = pd.read_csv(PIPELINE2_DIR / 'graph.csv')

print("Convertendo triplas e plotando Grafo da Pipeline 2 (Regressão Logística)...")
nodes_p2, edges_p2 = converter_triplas_para_nos_arestas(df_p2)

# Salva na mesma pasta recém-criada
output_file_p2 = str(Path('../../assets/html/knowledge_graph_pipeline2.html'))

plotar_grafo_projeto2(nodes_p2, edges_p2, output_html=output_file_p2)

Convertendo triplas e plotando Grafo da Pipeline 2 (Regressão Logística)...
../../assets/html/knowledge_graph_pipeline2.html
